# API 调用 OpenAI + Prompt 工程
样例来源： https://github.com/openai/openai-cookbook 

## API单论调用 实现单一任务
----
源代码： https://github.com/openai/openai-cookbook/blob/main/examples/completions_usage_api.ipynb

数据源： https://tianchi.aliyun.com/dataset/56 

### 1. AI 多轮对话 Assistant 设置角色


In [1]:
from openai import OpenAI

client = OpenAI()
 
messages = [
    {"role": "system", 
     "content": "你是一个数据分析助手，回答要专业、简洁"}
]

print("🤖 AI Chat 已启动（输入 exit 退出）")

while True:
    user_input = input("\n你：")

    if user_input.lower() == "exit":
        break
 
    messages.append({"role": "user", "content": user_input})

 
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages
    )

    reply = response.choices[0].message.content

    print("\nAI：", reply)

    messages.append({"role": "assistant", "content": reply})

🤖 AI Chat 已启动（输入 exit 退出）

AI： 今天是2024年4月27日。

AI： 抱歉，目前我的知识和系统时间仅支持到2024年6月，无法确认2026年的具体日期信息。

AI： 请提供具体的城市或地区名称，我可以帮您查询相关的天气信息。

AI： 抱歉，我无法实时获取洛杉矶的当前天气信息。您可以使用天气网站或应用（如Weather.com、AccuWeather）获取最新天气预报。

AI： 目前我无法提供洛杉矶的实时天气数据。建议您使用专业气象网站或应用，如Weather.com、AccuWeather或当地气象部门，获取最新的洛杉矶天气信息。

AI： 很抱歉听到你心情不好。如果愿意，可以告诉我具体原因，我愿意倾听并提供支持。适当休息、与朋友交流或者进行放松活动也有助于改善心情。


### 2. SQL 生成器

In [2]:
from openai import OpenAI
client = OpenAI()

def generate_mysql(question):
    prompt = f"""
你是一个电商广告数据分析师，请根据问题生成SQL。

数据库结构：

raw_sample(user_id, adgroup_id, clk, time_stamp)
ad_feature(adgroup_id, cate_id, brand, price)

user_profile(user_id, age_level, gender, occupation)
user_behavior_log(user_id, btag, cate, time_stamp)

字段说明：
- clk: 是否点击（1=点击，0=未点击）
- time_stamp: 时间戳

要求：
- 使用标准SQL
- 只返回MySQL，不要解释

问题：
{question}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content


In [3]:
generate_mysql("计算最近30天整体CTR")

'```sql\nSELECT \n    SUM(clk) / COUNT(*) AS ctr\nFROM \n    raw_sample\nWHERE \n    time_stamp >= UNIX_TIMESTAMP(DATE_SUB(CURDATE(), INTERVAL 30 DAY));\n```'

In [4]:
generate_mysql("统计每个广告的CTR，并按CTR排序")

'```sql\nSELECT\n    r.adgroup_id,\n    SUM(r.clk) / COUNT(*) AS ctr\nFROM\n    raw_sample r\nGROUP BY\n    r.adgroup_id\nORDER BY\n    ctr DESC;\n```'

### 3.文本摘要（NLP能力）

In [5]:
def summarize(text):
    prompt = f"""
请总结以下内容（90字以内）：

{text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

print(summarize("为什么OpenAI要给竞品做插件?很多朋友可能会懵，OpenAI不是有自己的Codex吗？为啥要给Anthropic的Claude Code做插件？" \
"答案很简单：因为开发者都在用Claude Code。你去问问身边写代码的朋友，十个有九个主力用的是Claude Code，Codex虽然强，但用户量就是干不过。" \
"Anthropic的Claude Code生态已经起来了，插件市场、MCP服务器、各种Skills，整个Agent化编程的基础设施都在Claude Code上跑得最顺。" \
"OpenAI这次的策略就是，既然打不过，那我就加入你。你用户多是吧？行，我直接把Codex的能力塞进你的工作流里，让你的用户也能用上我的模型。 " \
"这招太狠了，因为它不是竞争，而是渗透。这个插件到底能干啥？codex-plugin-cc 目前有几个核心命令，每一个都很实用：" \
"1. /codex:review - 常规代码审查就是让Codex以只读模式review你的代码，给你提意见、找bug、优化建议啥的。这个场景特别适合，Claude Code写完代码，你不太放心，想让另一个AI再看一眼。 " \
"2. /codex:adversarial-review - 对抗性审查这个更狠，是让Codex用一种挑刺的态度来审查代码。不是温和的建议，而是更aggressive的质疑：这段逻辑真的对吗？边界条件考虑了吗？性能会不会有问题？相当于给你的代码找个红队来攻击。" \
"3. /codex:rescue - 委派任务当Claude Code搞不定某个任务的时候，你可以直接把活儿扔给Codex，让它在后台跑。这个就是真正的打不过就叫队友。" \
"4. /codex:status 、 /codex:result 、 /codex:cancel - 后台任务管理既然可以委派任务，那就需要管理这些后台job。查状态、拿结果、取消任务，该有的都有了。整个设计思路就是，让Claude Code和Codex可以无缝协作"))

OpenAI为Anthropic的Claude Code开发插件，以借助其庞大用户生态，实现模型能力渗透。该插件支持代码审查、对抗性审查、任务委派与后台管理，促进Claude Code与Codex无缝协作，提升开发者体验。


### 4.情感分析-分类任务

In [7]:
def sentiment_analysis(text):
    prompt = f"""
请判断以下文本情感：
返回：positive / negative / neutral

文本：
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

print(sentiment_analysis("这个产品太差了"))

negative


### 5. 自动分类

In [14]:
def classify(text):
    prompt = f"""
请将文本分类为：
[投诉, 建议, 咨询，闲聊]

文本：
{text}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [15]:
print(classify('今天的洛杉矶天气真的好热啊，开车都要开空调。并且最近中东战争这么混乱，油价飞涨，让我天天开车开空调真的开始舍不得了。'))

文本类别：闲聊


### 6. Prompt + 结构化输出（输出json）

In [12]:
import json

def extract_info(df):
    prompt = f"""
从数据中提取：
- 顾客id
- 分类
- 消费等级
- 等其他你觉得有助于数据分析的特征

返回JSON格式

文本：
{df}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [13]:
print(extract_info("张三今天去了whole foods花了200多美元，买了很多organic 的食物。”\
                   “这些食物是很多中产家庭才能吃得起的，并且他的一个hot bar就吃了50多美元，真的是大手笔，他也是whole foods的大会员。"))

```json
{
  "顾客id": "张三",
  "分类": "食品购买",
  "消费等级": "高",
  "消费金额": {
    "总计": "200多美元",
    "hot bar": "50多美元"
  },
  "商品类型": "organic 食物",
  "会员级别": "whole foods 大会员",
  "客户群体定位": "中产家庭",
  "消费行为描述": "消费金额较高，大手笔购买，偏好有机食品"
}
```


### 7. 自动生成数据报告 

In [8]:
def build_data_info(df):
    return f"""
数据规模：
行数: {df.shape[0]}
列数: {df.shape[1]}

字段：
{list(df.columns)}

数据类型：
{df.dtypes}

缺失值：
{df.isnull().sum()}

示例数据：
{df.head(15).to_string()}
"""

def generate_report(df,desc):

    data_info = build_data_info(df)

    prompt = f"""
你是一个数据分析师。

请根据下面分析思路进行数据分析

请输出：
1. 总结数据结构，需要清洗的部分,df是导入的数据，desc是对这些数据的行列名称的介绍
2. 提出分析思路
3. 简单低根据数据特点进行1-2步的特征工程，指出关键指标
3. 可视化建议

数据描述：
{desc}

【数据结构与样本】
{data_info}
"""

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [11]:
import pandas as pd

df = pd.read_csv('../data/user_profile.csv')

with open('../data/data_description.txt', 'r', encoding='utf-8') as f:
    desc = f.read()

print(generate_report(df,desc))

好的，基于你提供的用户画像（user_profile）数据表及其描述，下面我从数据结构总结、数据清洗、分析思路、简单特征工程和可视化建议几个方面给出详细的分析方案：

---

### 1. 数据结构总结及需要清洗的部分

**数据结构（user_profile表）**

| 字段名                | 含义               | 数据类型   | 备注                |
|-----------------------|--------------------|------------|---------------------|
| userid                | 用户ID             | int64      | 主键，唯一标识用户   |
| cms_segid             | 微分组ID           | int64      | 用户分群标签         |
| cms_group_id          | cms组ID            | int64      | 用户分群标签         |
| final_gender_code     | 性别编码           | int64      | 1=男，2=女           |
| age_level             | 年龄等级           | int64      | 不同年龄区间等级     |
| pvalue_level          | 消费等级           | float64    | 1=低，2=中，3=高    |
| shopping_level        | 购物深度等级       | int64      | 1=浅，2=中，3=深    |
| occupation            | 职业（是否大学生） | int64      | 1=大学生，0=非大学生 |
| new_user_class_level  | 城市等级           | float64    | 城市等级，有缺失     |

> 样本量约106万行，9个特征列

**缺失值情况**

- `pvalue_level` 缺失量较大，约57.5万

## II. Prompt → System Design（系统设计）

## III. 单轮调用 → 多步推理（Agent 思维）

## IV. 静态知识 → 外部数据（RAG）

## VI. Demo → 产品级系统